# RadLite: Multi-Task Radiology AI on CPU

Run RadAssist (Qwen2.5-3B + Qwen3-4B with LoRA fine-tuning, quantized to GGUF Q4_K_M) directly in Google Colab.

**9 tasks supported:** RADS assignment, impression generation, temporal comparison, radiology NER, N/M staging, NLI, abnormality detection, and radiology QA.

| Task | Routed Model | Key Benchmark |
|------|-------------|---------------|
| RADS assignment | Qwen2.5-3B | Acc 0.770 |
| Impression generation | Qwen2.5-3B | ROUGE-L 0.502 |
| Temporal comparison | Qwen3-4B | Jaccard 0.923 |
| Radiology NER | Qwen3-4B | ROUGE-L 0.950 |
| N-staging | Qwen2.5-3B | Acc 0.890 |
| M-staging | Qwen2.5-3B | Acc 0.730 |
| Radiology NLI | Qwen2.5-3B | Acc 0.825 |
| Abnormality detection | Qwen3-4B | Per-label Acc 0.606 |
| Radiology QA | Qwen2.5-3B | ROUGE-L 0.107 |

## 1. Install Dependencies

In [ ]:
!pip install llama-cpp-python gdown -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 MB 10.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00


## 2. Download GGUF Models

Downloads the two quantized models (~1.8 GB + ~2.4 GB) from Google Drive.

**To use your own links:** Replace the `FILE_ID` values below with your Google Drive file IDs.

To get a file ID from a Google Drive share link:
- Share link looks like: `https://drive.google.com/file/d/XXXXXXXXXX/view`
- The file ID is the `XXXXXXXXXX` part.

In [ ]:
import os

MODELS_DIR = "/content/radlite_models"
os.makedirs(MODELS_DIR, exist_ok=True)

Q25_FILE_ID = "1Y6FOOVW-0y_h3OM5_FjojoxETdyZAbD3"
Q3_FILE_ID  = "1ghZ3DihYLrxClq1euSCalNvTfe8susFp"

Q25_PATH = os.path.join(MODELS_DIR, "qwen2.5-3b-radiology-Q4_K_M.gguf")
Q3_PATH  = os.path.join(MODELS_DIR, "qwen3-4b-radiology-Q4_K_M.gguf")

import gdown

for path, file_id, label in [
    (Q25_PATH, Q25_FILE_ID, "Qwen2.5-3B (1.8 GB)"),
    (Q3_PATH,  Q3_FILE_ID,  "Qwen3-4B  (2.4 GB)"),
]:
    if os.path.exists(path):
        print(f"{label}: already downloaded")
    else:
        print(f"Downloading {label}...")
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url, path, quiet=False)

print("\nReady!")

Downloading...
From (original): https://drive.google.com/uc?id=1Y6FOOVW-0y_h3OM5_FjojoxETdyZAbD3
From (redirected): https://drive.google.com/uc?id=1Y6FOOVW-0y_h3OM5_FjojoxETdyZAbD3&confirm=t&uuid=908aa76a-83cd-4896-8c6a-d96c5191c269
To: /content/radlite_models/qwen2.5-3b-radiology-Q4_K_M.gguf
100%|██████████| 1.93G/1.93G [00:28<00:00, 67.3MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1ghZ3DihYLrxClq1euSCalNvTfe8susFp
From (redirected): https://drive.google.com/uc?id=1ghZ3DihYLrxClq1euSCalNvTfe8susFp&confirm=t&uuid=f8042920-fb30-456e-99be-0588ea59bb00
To: /content/radlite_models/qwen3-4b-radiology-Q4_K_M.gguf
100%|██████████| 2.50G/2.50G [00:34<00:00, 71.7MB/s]


Ready!


## 3. Load Models & Define Task Router

In [ ]:
from llama_cpp import Llama
import re
import time

print("Loading Qwen2.5-3B (generation tasks)...")
model_q25 = Llama(
    model_path=Q25_PATH,
    n_ctx=2048,
    n_threads=4,
    verbose=False,
)

print("Loading Qwen3-4B (extraction tasks)...")
model_q3 = Llama(
    model_path=Q3_PATH,
    n_ctx=2048,
    n_threads=4,
    verbose=False,
)
print("Both models loaded.\n")

SYSTEM_PROMPT = "You are an expert radiologist. Follow the instructions and provide an accurate response."

TASK_ROUTES = {
    "rads_assignment":       "q25",
    "impression_generation": "q25",
    "radiology_nli":         "q25",
    "radiology_qa":          "q25",
    "n_staging":             "q25",
    "m_staging":             "q25",
    "temporal_comparison":   "q3",
    "abnormality_detection": "q3",
    "radiology_ner":         "q3",
}

TASK_PROMPTS = {
    "rads_assignment": (
        "[TASK: rads_assignment]\n"
        "Classify the following radiology report into the appropriate RADS category. "
        "Output ONLY the RADS category (e.g., 'BI-RADS 4', 'PI-RADS 3', 'TI-RADS 5').\n\n"
        "Report:\n{report}"
    ),
    "impression_generation": (
        "[TASK: impression_generation]\n"
        "Generate a concise clinical impression from the following radiology findings.\n\n"
        "Findings:\n{report}"
    ),
    "temporal_comparison": (
        "[TASK: temporal_comparison]\n"
        "Compare the following radiology findings and identify temporal changes "
        "(new, worsened, improved, resolved, unchanged).\n\n"
        "Findings:\n{report}"
    ),
    "radiology_ner": (
        "[TASK: radiology_ner]\n"
        "Extract all anatomical structures, observations (present/absent/uncertain), "
        "and change indicators from the following report.\n\n"
        "Report:\n{report}"
    ),
    "n_staging": (
        "[TASK: n_staging]\n"
        "Radiology Report: Lower thorax: Clear. Lymph nodes: Multiple enlarged "
        "retroperitoneal and para-aortic lymph nodes, largest 2.3 cm. IMPRESSION: "
        "1. Retroperitoneal lymphadenopathy. Lymph node involvement: Positive. "
        "N staging: N2\n\n"
        "Radiology Report: Lower thorax: Clear. Liver: Normal. Lymph nodes: No "
        "lymphadenopathy. IMPRESSION: 1. No acute findings. Lymph node involvement: "
        "Negative. N staging: N0\n\n"
        "Radiology Report: {report}\n\n"
        "Lymph node involvement:"
    ),
    "m_staging": (
        "[TASK: m_staging]\n"
        "Radiology Report: {report}\n\n"
        "Determine the M staging (metastatic involvement):"
    ),
    "radiology_nli": (
        "[TASK: radiology_nli]\n"
        "{report}\n\n"
        "Determine if the hypothesis is entailed, contradicted, or neutral given the premise:"
    ),
    "abnormality_detection": (
        "[TASK: abnormality_detection]\n"
        "Classify each finding as positive, negative, or uncertain: "
        "atelectasis, cardiomegaly, consolidation, edema, enlarged cardiomediastinum, "
        "fracture, lung lesion, lung opacity, pleural effusion, pleural other, pneumonia, "
        "pneumothorax, support devices, no finding.\n\n"
        "Report:\n{report}"
    ),
    "radiology_qa": (
        "[TASK: radiology_qa]\n"
        "Answer the following radiology question.\n\n"
        "Question:\n{report}"
    ),
}

TASK_MAX_TOKENS = {
    "rads_assignment": 30,
    "n_staging": 64,
    "m_staging": 10,
    "radiology_nli": 10,
}

def detect_task(text):
    explicit = re.search(r'\[TASK:\s*([a-z_]+)\]', text, re.IGNORECASE)
    if explicit:
        task = explicit.group(1).lower()
        if task in TASK_ROUTES:
            return task
    return "radiology_qa"

def run_radlite(report, task=None):
    if task is None:
        task = detect_task(report)

    model_key = TASK_ROUTES.get(task, "q25")
    model = model_q25 if model_key == "q25" else model_q3
    model_name = "Qwen2.5-3B" if model_key == "q25" else "Qwen3-4B"
    prompt = TASK_PROMPTS.get(task, TASK_PROMPTS["radiology_qa"]).format(report=report)
    max_tokens = TASK_MAX_TOKENS.get(task, 256)

    full_prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    t0 = time.time()
    response = model(
        full_prompt,
        max_tokens=max_tokens,
        temperature=0.1,
        stop=["<|im_end|>"],
    )
    elapsed = time.time() - t0
    text = response["choices"][0]["text"].strip()
    tokens = response["usage"]["completion_tokens"]
    tok_s = tokens / elapsed if elapsed > 0 else 0

    return {
        "task": task,
        "model": model_name,
        "response": text,
        "tokens": tokens,
        "time_s": round(elapsed, 1),
        "tok_per_s": round(tok_s, 1),
    }

Loading Qwen2.5-3B (generation tasks)...


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Loading Qwen3-4B (extraction tasks)...


llama_context: n_ctx_seq (2048) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


Both models loaded.



## 4. Example Reports for All 9 Tasks

In [ ]:
EXAMPLES = {
    "1. RADS Assignment": {
        "task": "rads_assignment",
        "report": (
            "Breast ultrasound shows a 2.1 cm irregular hypoechoic mass with angular "
            "margins and posterior shadowing in the upper outer quadrant of the right "
            "breast. No associated lymphadenopathy."
        ),
    },
    "2. Impression Generation": {
        "task": "impression_generation",
        "report": (
            "Chest PA and lateral radiographs: The heart is normal in size. The lungs are "
            "clear bilaterally. No pleural effusion, pneumothorax, or focal consolidation. "
            "The osseous structures are intact. No acute cardiopulmonary abnormality."
        ),
    },
    "3. Temporal Comparison": {
        "task": "temporal_comparison",
        "report": (
            "Current: Chest radiograph shows a 2.1 cm nodular opacity in the right lower "
            "lobe, new compared to prior. A small right pleural effusion is present, "
            "slightly increased from prior. Left lung remains clear. Heart size is stable. "
            "Prior: Chest radiograph from 3 months ago showed clear lungs bilaterally "
            "with no pleural effusion."
        ),
    },
    "4. Radiology NER": {
        "task": "radiology_ner",
        "report": (
            "CT chest with contrast: A 3.2 cm spiculated mass is present in the right upper "
            "lobe with associated right hilar lymphadenopathy. A 0.8 cm ground-glass nodule "
            "is seen in the left lower lobe. Small bilateral pleural effusions are present. "
            "Heart size is normal."
        ),
    },
    "5. N-Staging": {
        "task": "n_staging",
        "report": (
            "CT chest abdomen pelvis: Right upper lobe 3.5 cm mass with biopsy-proven "
            "adenocarcinoma. Right hilar lymph nodes measuring 1.8 cm. Subcarinal lymph "
            "node measuring 1.2 cm. No contralateral mediastinal lymphadenopathy. No "
            "supraclavicular lymphadenopathy. Liver, adrenal glands, and bones appear "
            "unremarkable."
        ),
    },
    "6. M-Staging": {
        "task": "m_staging",
        "report": (
            "CT chest abdomen pelvis: Known left lower lobe adenocarcinoma. Multiple new "
            "hepatic metastases, largest 3.1 cm in the right lobe. Bilateral adrenal "
            "metastases. Multiple sclerotic vertebral body lesions consistent with osseous "
            "metastases."
        ),
    },
    "7. Radiology NLI": {
        "task": "radiology_nli",
        "report": (
            "Premise: Chest radiograph shows no acute cardiopulmonary abnormality. "
            "Hypothesis: The patient has pneumonia."
        ),
    },
    "8. Abnormality Detection": {
        "task": "abnormality_detection",
        "report": (
            "Chest radiograph: Bilateral perihilar opacities consistent with pulmonary "
            "edema. Cardiomegaly is present. Small bilateral pleural effusions. No "
            "pneumothorax. The osseous structures are intact."
        ),
    },
    "9. Radiology QA": {
        "task": "radiology_qa",
        "report": (
            "What is the most common imaging finding in acute pulmonary embolism on CT "
            "pulmonary angiography?"
        ),
    },
}

print(f"Loaded {len(EXAMPLES)} example reports.")

Loaded 9 example reports.


## 5. Run All Examples

In [ ]:
for title, ex in EXAMPLES.items():
    result = run_radlite(ex["report"], task=ex["task"])
    print(f"{'='*60}")
    print(f"{title}")
    print(f"  Task:   {result['task']}")
    print(f"  Model:  {result['model']} (routed)")
    print(f"  Time:   {result['time_s']}s ({result['tok_per_s']} tok/s)")
    print(f"  Output: {result['response']}")
    print()

1. RADS Assignment
  Task:   rads_assignment
  Model:  Qwen2.5-3B (routed)
  Time:   11.0s (0.5 tok/s)
  Output: BI-RADS 4

2. Impression Generation
  Task:   impression_generation
  Model:  Qwen2.5-3B (routed)
  Time:   8.5s (0.7 tok/s)
  Output: Examination within normal limits.

3. Temporal Comparison
  Task:   temporal_comparison
  Model:  Qwen3-4B (routed)
  Time:   30.8s (1.8 tok/s)
  Output: <think>

</think>

- [New Finding] new (related to: opacity)
- [Worsened] increased (related to: effusion)
- [No Change] remains (related to: clear)
- [No Change] stable (related to: general)

4. Radiology NER
  Task:   radiology_ner
  Model:  Qwen3-4B (routed)
  Time:   34.5s (2.1 tok/s)
  Output: <think>

</think>

Anatomy: right; upper; lobe; right; hilar; left; lower; lobe; bilateral; pleural; Heart; size
Observation (Present): 3.2 cm; spiculated; mass; lymphadenopathy; 0.8 cm; ground-glass; nodule; Small; effusions; normal

5. N-Staging
  Task:   n_staging
  Model:  Qwen2.5-3B (routed)


## 6. Playground — Try Your Own Report

Edit the report text below and re-run the cell.

In [ ]:
# @title Run RadLite on a custom report { run: "auto" }
task = "rads_assignment" # @param ["rads_assignment", "impression_generation", "temporal_comparison", "radiology_ner", "n_staging", "m_staging", "radiology_nli", "abnormality_detection", "radiology_qa"]
report = "Prostate MRI demonstrates a 1.5 cm focal lesion in the left peripheral zone with marked diffusion restriction and low T2 signal. No extracapsular extension." # @param {type: "string"}

result = run_radlite(report, task=task)
print(f"Task:   {result['task']}")
print(f"Model:  {result['model']} (routed)")
print(f"Time:   {result['time_s']}s ({result['tok_per_s']} tok/s)")
print(f"\n{result['response']}")

Task:   rads_assignment
Model:  Qwen2.5-3B (routed)
Time:   8.9s (0.6 tok/s)

PI-RADS 5


## 7. Challenge Cases — Edge Cases and Cross-System Traps

In [ ]:
CHALLENGES = [
    {
        "label": "Cross-system trap (liver + lung nodule)",
        "task": "rads_assignment",
        "report": (
            "62-year-old male with chronic hepatitis C. Multiphasic MRI liver: "
            "A 2.8 cm lesion in segment 5 demonstrating arterial phase hyperenhancement "
            "with washout on portal venous phase and delayed phase capsular retraction. "
            "Restricted diffusion present. No enhancing pseudocapsule. Background cirrhotic "
            "liver with splenomegaly. Chest radiograph from 2 weeks ago showed a 1.2 cm "
            "solid nodule in the right upper lobe, under surveillance."
        ),
        "expected": "LI-RADS 5",
    },
    {
        "label": "Borderline BI-RADS (4A vs 4B)",
        "task": "rads_assignment",
        "report": (
            "58-year-old female, history of right mastectomy. Left breast ultrasound: "
            "9 mm oval circumscribed parallel hypoechoic mass in upper outer quadrant, "
            "no internal vascularity. A second 6 mm irregular hypoechoic mass, not "
            "parallel, indistinct margins, with internal vascularity on Doppler. "
            "Heterogeneously dense breasts. Normal left axillary lymph node."
        ),
        "expected": "BI-RADS 4B",
    },
    {
        "label": "No applicable RADS (bladder lesion trap)",
        "task": "rads_assignment",
        "report": (
            "52-year-old male with microscopic hematuria. CT urography: Kidneys normal. "
            "Along the left posterior bladder wall, a 1.4 cm flat plaque-like area of "
            "focal mucosal irregularity with mild enhancement on delayed phase, new "
            "compared to prior CT 14 months ago. No perivesical fat stranding or "
            "lymphadenopathy. 25 pack-year smoking history."
        ),
        "expected": "No applicable RADS",
    },
    {
        "label": "Complex temporal with mixed changes",
        "task": "temporal_comparison",
        "report": (
            "Current CT: Right upper lobe mass decreased from 3.5 cm to 2.8 cm after "
            "chemotherapy. New 0.6 cm ground-glass nodule in left lower lobe, not present "
            "on prior. Right pleural effusion resolved. Stable mediastinal lymphadenopathy. "
            "New sclerotic focus at L2 vertebral body."
        ),
        "expected": "Mixed: improved (RUL mass), new (LLL nodule, L2 lesion), resolved (effusion), unchanged (lymphadenopathy)",
    },
]

for c in CHALLENGES:
    result = run_radlite(c["report"], task=c["task"])
    print(f"{'='*60}")
    print(f"Challenge: {c['label']}")
    print(f"  Expected: {c['expected']}")
    print(f"  Model:    {result['model']}")
    print(f"  Output:   {result['response']}")
    print()

---

**RadLite** — Gupta & Bose (2026). *Multi-Task LoRA Fine-Tuning of Small Language Models for CPU-Deployable Radiology AI.*

Models: Qwen2.5-3B-Instruct + Qwen3-4B, fine-tuned with LoRA (rank=64, alpha=128) on 161,586 samples across 9 tasks from 12 public datasets.